# NB04. The Independence Finding: R² ≠ AdvTop-k

**목적**: NB03 결과로부터 C1의 핵심 발견 — 예측 충실도(R²)가 귀인 충실도(AdvTop-k)를 보장하지 않음 — 을 정량적으로 실증한다.

**Dependencies**: NB03 (results.json)

**Outputs**: Figure 2 (R² vs AdvTop-4 scatter), 통계 검정 결과

**핵심 질문**: R²와 AdvTop-4 사이에 유의한 상관이 있는가? 있다면 R²만으로 AdvTop-4를 예측할 수 있는 수준인가?

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

NB03_DIR = Path('../outputs/NB03')
OUT_DIR = Path('../outputs/NB04')
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(NB03_DIR / 'results.json') as f:
    results = json.load(f)

# Flatten to DataFrame
rows = []
for data_name in results:
    for method, m in results[data_name].items():
        rows.append({
            'Dataset': data_name, 'Method': method,
            'R2': m['R2'], 'Spearman': m['Spearman'],
            'AdvTop1': m['AdvTop1'], 'AdvTop4': m['AdvTop4'],
            'AdvFull_R': m['AdvFull_R'],
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

Dataset       Method     R2  Spearman  AdvTop1  AdvTop4  AdvFull_R
   GMSC      Tree-d1 0.9362    0.9705   0.8077   0.9303     0.9027
   GMSC Tree-d1+mono 0.9270    0.9639   0.8030   0.9164     0.9094
   GMSC      Tree-d3 0.9787    0.9874   0.9060   0.9506     0.9670
   GMSC      Tree-d6 0.9898    0.9935   0.9547   0.9663     0.9638
   GMSC          EBM 0.9395    0.9725   0.8423   0.9273     0.9622
   GMSC     EBM+mono 0.8792    0.9451   0.6637   0.9219     0.9019
   GMSC        Ridge 0.3181    0.5921   0.3053   0.5462     0.6652
   GMSC OptBin+Ridge 0.9096    0.9529   0.8173   0.8782     0.9338
     HC      Tree-d1 0.9024    0.9523   0.9639   0.8237     0.6734
     HC Tree-d1+mono 0.8685    0.9338   0.9350   0.6087     0.6804
     HC      Tree-d3 0.9524    0.9758   0.9673   0.8794     0.8891
     HC      Tree-d6 0.9720    0.9854   0.9748   0.9108     0.9133
     HC          EBM 0.9263    0.9633   0.9371   0.8436     0.8849
     HC     EBM+mono 0.8336    0.9184   0.9262   0.5456     0.

## 1. R² vs AdvTop-4: 상관 분석

**가설**: R²가 높으면 AdvTop-4도 높을까?
**반례**: OptBin+Ridge는 R²=0.86(HC)이면서 AdvTop-4=0.62. Tree-d1은 R²=0.90에서 AdvTop-4=0.82.

In [2]:
# Per-dataset and pooled correlation
for subset_name, subset in [('GMSC', df[df.Dataset=='GMSC']),
                             ('HC', df[df.Dataset=='HC']),
                             ('Pooled', df)]:
    rho, pval = spearmanr(subset['R2'], subset['AdvTop4'])
    r2_of_r2 = np.corrcoef(subset['R2'], subset['AdvTop4'])[0, 1] ** 2
    print(f'{subset_name:8s}: Spearman(R², AT4) = {rho:.3f} (p={pval:.4f}), '
          f'R²(R²→AT4) = {r2_of_r2:.3f}')

print('\n--- 핵심 반례 ---')
for data_name in ['GMSC', 'HC']:
    sub = df[df.Dataset == data_name]
    for _, row in sub.iterrows():
        if row['Method'] in ['Tree-d1', 'OptBin+Ridge', 'Ridge', 'EBM']:
            flag = '⚠️' if row['R2'] > 0.8 and row['AdvTop4'] < 0.7 else ''
            print(f'  {data_name} {row["Method"]:16s}  R²={row["R2"]:.3f}  AT4={row["AdvTop4"]:.3f}  {flag}')

GMSC    : Spearman(R², AT4) = 0.905 (p=0.0020), R²(R²→AT4) = 0.985
HC      : Spearman(R², AT4) = 0.881 (p=0.0039), R²(R²→AT4) = 0.908
Pooled  : Spearman(R², AT4) = 0.856 (p=0.0000), R²(R²→AT4) = 0.435

--- 핵심 반례 ---
  GMSC Tree-d1           R²=0.936  AT4=0.930  
  GMSC EBM               R²=0.940  AT4=0.927  
  GMSC Ridge             R²=0.318  AT4=0.546  
  GMSC OptBin+Ridge      R²=0.910  AT4=0.878  
  HC Tree-d1           R²=0.902  AT4=0.824  
  HC EBM               R²=0.926  AT4=0.844  
  HC Ridge             R²=0.829  AT4=0.616  ⚠️
  HC OptBin+Ridge      R²=0.863  AT4=0.622  ⚠️


## 2. 방법 유형별 분석

선형(Ridge, OptBin+Ridge) vs 트리(Tree-d1/3/6) vs GAM(EBM) 계열로 구분하여, 구조적 실패가 선형 회귀 계열에 집중되는지 확인한다.

In [3]:
family_map = {
    'Tree-d1': 'Tree', 'Tree-d1+mono': 'Tree', 'Tree-d3': 'Tree', 'Tree-d6': 'Tree',
    'EBM': 'GAM', 'EBM+mono': 'GAM',
    'Ridge': 'Linear', 'OptBin+Ridge': 'Linear',
}
df['Family'] = df['Method'].map(family_map)

print('=== AdvTop-4 by Method Family ===')
for data_name in ['GMSC', 'HC']:
    sub = df[df.Dataset == data_name]
    print(f'\n{data_name}:')
    for fam in ['Tree', 'GAM', 'Linear']:
        fsub = sub[sub.Family == fam]
        print(f'  {fam:8s}  R²: {fsub.R2.mean():.3f} [{fsub.R2.min():.3f}-{fsub.R2.max():.3f}]  '
              f'AT4: {fsub.AdvTop4.mean():.3f} [{fsub.AdvTop4.min():.3f}-{fsub.AdvTop4.max():.3f}]')

print('\n=== 핵심 관찰 ===')
print('HC에서 Linear 계열은 R² 0.83-0.86 범위에서 AT4 0.62로,')
print('Tree 계열의 R² 0.87-0.97 / AT4 0.61-0.91과 비교하면')
print('R²가 유사하더라도 AT4가 크게 다를 수 있음을 보여줌.')

=== AdvTop-4 by Method Family ===

GMSC:
  Tree      R²: 0.958 [0.927-0.990]  AT4: 0.941 [0.916-0.966]
  GAM       R²: 0.909 [0.879-0.940]  AT4: 0.925 [0.922-0.927]
  Linear    R²: 0.614 [0.318-0.910]  AT4: 0.712 [0.546-0.878]

HC:
  Tree      R²: 0.924 [0.869-0.972]  AT4: 0.806 [0.609-0.911]
  GAM       R²: 0.880 [0.834-0.926]  AT4: 0.695 [0.546-0.844]
  Linear    R²: 0.846 [0.829-0.863]  AT4: 0.619 [0.616-0.622]

=== 핵심 관찰 ===
HC에서 Linear 계열은 R² 0.83-0.86 범위에서 AT4 0.62로,
Tree 계열의 R² 0.87-0.97 / AT4 0.61-0.91과 비교하면
R²가 유사하더라도 AT4가 크게 다를 수 있음을 보여줌.


## 3. 결론

**C1 지지 여부**: NB03의 8개 방법 비교에서 R²와 AdvTop-4의 관계를 분석한 결과를 정리한다.

In [4]:
# Save results summary
df.to_csv(OUT_DIR / 'independence_analysis.csv', index=False)
print(f'Saved to {OUT_DIR}/')

Saved to ..\outputs\NB04/
